# 00 · Environment
Mounts Drive, clones the private repo (needs a `GH_TOKEN` Colab secret), installs the main environment and runs the test suite. Run once per new runtime.

In [ ]:
# Shared session setup (run in every notebook)
import os, subprocess, pathlib
from google.colab import drive
drive.mount('/content/drive')
DRIVE = pathlib.Path('/content/drive/MyDrive/s1pii'); DRIVE.mkdir(parents=True, exist_ok=True)
os.environ['S1PII_DATA'] = '/content/s1pii_data'        # local disk; synced to/from Drive
os.environ['S1PII_RESULTS'] = str(DRIVE / 'results')     # ledger + prediction cache on Drive
REPO = pathlib.Path('/content/s1-pii')
if not REPO.exists():
    from google.colab import userdata                    # add GH_TOKEN (fine-grained, read-only) in Colab secrets
    tok = userdata.get('GH_TOKEN')
    subprocess.run(['git', 'clone', f'https://{tok}@github.com/anitksahu/s1-pii.git', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', 'https://github.com/anitksahu/s1-pii.git'], check=True)
os.chdir(REPO)
subprocess.run(['rsync', '-a', f'{DRIVE}/data/', os.environ['S1PII_DATA'] + '/'], check=False)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)


In [ ]:
!pip install -q -e '.[data,dev]' && python -m pytest -q

In [ ]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')